# NAM (Neural Amp Modeler) Pure Python+NumPy Inference

This notebook implements NAM model inference using only Python and NumPy — no PyTorch.
We test with 3 models from the NeuralAmpModelerCore example models:

1. **LSTM tiny** — 3 hidden units, 1 layer (2.3 KB)
2. **WaveNet tiny** — 2 layer arrays, small channels (3.9 KB)
3. **WaveNet standard** — 2 layer arrays, 16/8 channels, 10 dilations each (407 KB)

Goal: assess whether pure Python+NumPy can run NAM inference fast enough for real-time audio.

In [ ]:
import json
import time
import numpy as np
import cProfile
import pstats
import io

SAMPLE_RATE = 48000
print(f"NumPy version: {np.__version__}")

## 1. Weight Reader + Core Ops

In [ ]:
class WeightReader:
    def __init__(self, weights):
        self.weights = np.array(weights, dtype=np.float32)
        self.offset = 0
    def read(self, count):
        result = self.weights[self.offset:self.offset + count]
        self.offset += count
        return result
    def read_conv1d(self, out_ch, in_ch, kernel_size=1, has_bias=True):
        w = self.read(out_ch * in_ch * kernel_size).reshape(out_ch, in_ch, kernel_size)
        b = self.read(out_ch) if has_bias else None
        return w, b
    def read_conv1x1(self, out_ch, in_ch, has_bias=True):
        w = self.read(out_ch * in_ch).reshape(out_ch, in_ch)
        b = self.read(out_ch) if has_bias else None
        return w, b
    @property
    def remaining(self):
        return len(self.weights) - self.offset


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -88, 88)))


def conv1d(x, weight, bias=None, dilation=1):
    """Causal 1D conv: x (C_in, L), weight (C_out, C_in, K) -> (C_out, L')."""
    C_out, C_in, K = weight.shape
    L = x.shape[1]
    L_out = L - (K - 1) * dilation
    if L_out <= 0:
        return np.zeros((C_out, 0), dtype=np.float32)
    cols = np.zeros((C_in * K, L_out), dtype=np.float32)
    for k in range(K):
        offset = (K - 1 - k) * dilation
        cols[k * C_in:(k + 1) * C_in, :] = x[:, offset:offset + L_out]
    out = weight.reshape(C_out, C_in * K) @ cols
    if bias is not None:
        out += bias[:, np.newaxis]
    return out


def conv1x1(x, weight, bias=None):
    out = weight @ x
    if bias is not None:
        out += bias[:, np.newaxis]
    return out


ACTIVATIONS = {'Tanh': np.tanh, 'ReLU': lambda x: np.maximum(0, x), 'Sigmoid': sigmoid}

## 2. LSTM Implementation

In [ ]:
class LSTMCell:
    def __init__(self, input_size, hidden_size, W, b, initial_h, initial_c):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.W, self.b = W, b
        self._initial_h, self._initial_c = initial_h.copy(), initial_c.copy()
        self.h, self.c = initial_h.copy(), initial_c.copy()

    def reset(self):
        self.h, self.c = self._initial_h.copy(), self._initial_c.copy()

    def process(self, x):
        H = self.hidden_size
        ifgo = self.W @ np.concatenate([x, self.h]) + self.b
        i, f, g, o = sigmoid(ifgo[:H]), sigmoid(ifgo[H:2*H]), np.tanh(ifgo[2*H:3*H]), sigmoid(ifgo[3*H:])
        self.c = f * self.c + i * g
        self.h = o * np.tanh(self.c)
        return self.h


class LSTMModel:
    def __init__(self, config, reader):
        self.hidden_size = H = config['hidden_size']
        self.cells = []
        for i in range(config.get('num_layers', 1)):
            in_sz = config.get('input_size', 1) if i == 0 else H
            W = reader.read(4*H*(in_sz+H)).reshape(4*H, in_sz+H)
            b = reader.read(4*H)
            self.cells.append(LSTMCell(in_sz, H, W, b, reader.read(H), reader.read(H)))
        self.head_w = reader.read(H).reshape(1, H)
        self.head_b = reader.read(1)
        self.receptive_field = 1

    def reset(self):
        for c in self.cells: c.reset()

    def process(self, audio):
        output = np.zeros_like(audio)
        for t in range(len(audio)):
            x = np.array([audio[t]], dtype=np.float32)
            for cell in self.cells:
                x = cell.process(x)
            output[t] = (self.head_w @ x + self.head_b)[0]
        return output

## 3. WaveNet Implementation

In [ ]:
class WaveNetLayer:
    def __init__(self, channels, condition_size, kernel_size, dilation, activation,
                 gated, reader, has_layer1x1=True, has_head1x1=False, head1x1_out=1):
        self.channels, self.dilation, self.gated = channels, dilation, gated
        mid_ch = 2 * channels if gated else channels
        self.act_fn = ACTIVATIONS.get(activation, np.tanh)
        self.conv_w, self.conv_b = reader.read_conv1d(mid_ch, channels, kernel_size)
        self.mix_w, _ = reader.read_conv1x1(mid_ch, condition_size, has_bias=False)
        self.has_layer1x1 = has_layer1x1
        if has_layer1x1:
            self.l1x1_w, _ = reader.read_conv1x1(channels, channels, has_bias=False)
        self.has_head1x1 = has_head1x1
        if has_head1x1:
            self.h1x1_w, _ = reader.read_conv1x1(head1x1_out, channels, has_bias=False)

    def forward(self, x, condition, out_length):
        z_conv = conv1d(x, self.conv_w, self.conv_b, dilation=self.dilation)
        z_mix = conv1x1(condition, self.mix_w)
        L = min(z_conv.shape[1], z_mix.shape[1])
        z = z_conv[:, -L:] + z_mix[:, -L:]
        z_act = (np.tanh(z[:self.channels]) * sigmoid(z[self.channels:])) if self.gated else self.act_fn(z)
        layer_out = conv1x1(z_act, self.l1x1_w) if self.has_layer1x1 else z_act
        head_out = conv1x1(z_act, self.h1x1_w)[:, -out_length:] if self.has_head1x1 else z_act[:, -out_length:]
        return x[:, -layer_out.shape[1]:] + layer_out, head_out


class WaveNetLayerArray:
    def __init__(self, config, reader):
        channels = config['channels']
        ks = config.get('kernel_size', 3)
        dilations = config['dilations']
        act = config.get('activation', 'Tanh')
        gated = config.get('gated', False)
        l1x1 = config.get('layer1x1', None)
        h1x1 = config.get('head1x1', None)
        has_l1x1 = l1x1.get('active', True) if isinstance(l1x1, dict) else True
        has_h1x1 = h1x1.get('active', False) if isinstance(h1x1, dict) else False
        h1x1_out = h1x1.get('out_channels', 1) if isinstance(h1x1, dict) else 1
        self.receptive_field = 1 + sum((ks - 1) * d for d in dilations)
        self.rechannel_w, _ = reader.read_conv1x1(channels, config['input_size'], has_bias=False)
        self.layers = [WaveNetLayer(channels, config['condition_size'], ks, d, act, gated, reader,
                                    has_l1x1, has_h1x1, h1x1_out) for d in dilations]
        skip_ch = h1x1_out if has_h1x1 else channels
        self.head_rechannel_w, self.head_rechannel_b = reader.read_conv1x1(
            config['head_size'], skip_ch, has_bias=config.get('head_bias', False))

    def forward(self, x, condition, head_input=None):
        out_length = min(x.shape[1], condition.shape[1]) - (self.receptive_field - 1)
        x = conv1x1(x, self.rechannel_w)
        for layer in self.layers:
            x, ht = layer.forward(x, condition, out_length)
            head_input = ht if head_input is None else head_input[:, -out_length:] + ht
        return conv1x1(head_input, self.head_rechannel_w, self.head_rechannel_b), x


class WaveNetModel:
    def __init__(self, config, reader):
        self.condition_dsp = None
        if config.get('condition_dsp'):
            cd = config['condition_dsp']
            self.condition_dsp = WaveNetModel(cd.get('config', cd),
                WeightReader(cd['weights']) if 'weights' in cd else reader)
        self.layer_arrays = [WaveNetLayerArray(la, reader) for la in config['layers']]
        self.head_layers = None
        if config.get('head'):
            hc = config['head']
            self.head_layers = []
            act = ACTIVATIONS.get(hc.get('activation', 'ReLU'), lambda x: np.maximum(0, x))
            in_ch = hc.get('in_channels', config['layers'][-1]['head_size'])
            for i in range(hc['num_layers']):
                o = hc.get('out_channels', 1) if i == hc['num_layers']-1 else hc['channels']
                w, b = reader.read_conv1x1(o, in_ch if i == 0 else hc['channels'], has_bias=True)
                self.head_layers.append((act, w, b))
        self.head_scale = reader.read(1)[0] if reader.remaining >= 1 else config.get('head_scale', 1.0)
        self.receptive_field = 1 + sum(la.receptive_field - 1 for la in self.layer_arrays)

    def process(self, audio):
        x = audio[np.newaxis, :]
        cond = self.condition_dsp.process_2d(x) if self.condition_dsp else x
        head_in, y = None, x
        for la in self.layer_arrays:
            head_in, y = la.forward(y, cond, head_in)
        result = self.head_scale * head_in
        if self.head_layers:
            for act, w, b in self.head_layers:
                result = conv1x1(act(result), w, b)
        return result[0]

    def process_2d(self, x):
        cond = self.condition_dsp.process_2d(x) if self.condition_dsp else x
        head_in, y = None, x
        for la in self.layer_arrays:
            head_in, y = la.forward(y, cond, head_in)
        return self.head_scale * head_in

## 4. Load Models + Test Audio

In [ ]:
def load_nam(path):
    data = json.load(open(path))
    reader = WeightReader(data['weights'])
    model = LSTMModel(data['config'], reader) if data['architecture'] == 'LSTM' else WaveNetModel(data['config'], reader)
    print(f"Loaded {data['architecture']} from {path} ({reader.offset}/{len(reader.weights)} weights)")
    return model

lstm = load_nam('lstm_tiny.nam')
wn_tiny = load_nam('wavenet_tiny.nam')
wn_std = load_nam('wavenet_standard.nam')

def make_audio(dur=1.0, freq=440.0):
    t = np.arange(int(SAMPLE_RATE * dur), dtype=np.float32) / SAMPLE_RATE
    sig = (0.5*np.sin(2*np.pi*freq*t) + 0.3*np.sin(4*np.pi*freq*t) +
           0.1*np.sin(6*np.pi*freq*t) + 0.05*np.sin(10*np.pi*freq*t)).astype(np.float32)
    return sig * np.exp(-t * 2.0).astype(np.float32)

test_1s = make_audio(1.0)
print(f"Test signal: {len(test_1s)} samples @ {SAMPLE_RATE} Hz")

## 5. Correctness Tests

In [ ]:
def test_model(name, model, audio):
    if isinstance(model, LSTMModel): model.reset()
    out = model.process(audio)
    rf = model.receptive_field
    expected = len(audio) if isinstance(model, LSTMModel) else len(audio) - (rf - 1)
    print(f"{name}: in={len(audio)}, out={len(out)} (expected {expected}), "
          f"range=[{out.min():.4f}, {out.max():.4f}], rms={np.sqrt(np.mean(out**2)):.4f}"
          f"{f', rf={rf}' if rf > 1 else ''}")
    assert len(out) == expected
    assert not np.any(np.isnan(out)) and not np.all(out == 0)
    # Silence test
    sil = np.zeros(max(2048, rf + 512), dtype=np.float32)
    if isinstance(model, LSTMModel): model.reset()
    model.process(sil)
    print(f"  PASSED")

short = make_audio(0.2)
test_model("LSTM tiny (H=3)", lstm, short)
test_model("WaveNet tiny", wn_tiny, short)
test_model("WaveNet standard", wn_std, short)

## 6. Batch Performance (1 second of audio)

In [ ]:
def bench_batch(name, model, audio, n=3):
    dur = len(audio) / SAMPLE_RATE
    times = []
    for _ in range(n):
        if isinstance(model, LSTMModel): model.reset()
        t0 = time.perf_counter()
        model.process(audio)
        times.append(time.perf_counter() - t0)
    avg = np.mean(times)
    rtf = avg / dur
    print(f"  {name:<30} {avg*1000:>8.1f} ms  {rtf:>6.3f}x  "
          f"{len(audio)/avg:>10.0f} samp/s  {'OK' if rtf < 1 else 'TOO SLOW'}")
    return {'name': name, 'avg_ms': avg*1000, 'rtf': rtf}

print(f"{'Model':<32} {'Time':>8}  {'RTF':>6}  {'Throughput':>10}")
print("-" * 72)
results = [
    bench_batch("LSTM tiny (H=3)", lstm, test_1s),
    bench_batch("WaveNet tiny", wn_tiny, test_1s),
    bench_batch("WaveNet standard (16ch)", wn_std, test_1s),
]

## 7. Per-Buffer Latency (Simulated DAW Callbacks)

In [ ]:
def bench_buffers(name, model, sizes=[64, 128, 256, 512, 1024, 2048]):
    print(f"\n  {name}")
    print(f"  {'Buf':>6} {'Time(ms)':>10} {'Budget':>10} {'Usage':>8}")
    for bs in sizes:
        budget = bs / SAMPLE_RATE * 1000
        rf = model.receptive_field
        n = bs + (rf - 1) if not isinstance(model, LSTMModel) else bs
        buf = make_audio(n / SAMPLE_RATE)[:n]
        if isinstance(model, LSTMModel):
            model.reset()
            model.process(np.zeros(bs, dtype=np.float32))  # warmup
        times = []
        for _ in range(20):
            t0 = time.perf_counter()
            model.process(buf)
            times.append(time.perf_counter() - t0)
        avg_ms = np.mean(times) * 1000
        pct = avg_ms / budget * 100
        print(f"  {bs:>6} {avg_ms:>10.2f} {budget:>10.2f} {pct:>7.0f}% {'OK' if pct < 100 else 'OVER'}")

bench_buffers("LSTM tiny (H=3)", lstm)
bench_buffers("WaveNet tiny", wn_tiny)
bench_buffers("WaveNet standard (16ch)", wn_std)

## 8. cProfile — Hot Spots

In [ ]:
def profile(name, model, audio):
    if isinstance(model, LSTMModel): model.reset()
    pr = cProfile.Profile()
    pr.enable()
    model.process(audio)
    pr.disable()
    s = io.StringIO()
    pstats.Stats(pr, stream=s).sort_stats('cumulative').print_stats(8)
    print(f"--- {name} ---")
    print(s.getvalue())

profile("LSTM tiny (1s)", lstm, test_1s)
profile("WaveNet standard (1s)", wn_std, test_1s)

## 9. LSTM Scaling: Hidden Size vs Performance

Real NAM LSTM models use hidden_size=24-40. How does performance scale?

In [ ]:
quarter = make_audio(0.25)
print(f"{'H':>4} {'1s (ms)':>10} {'RTF':>8} {'128-buf (ms)':>14} {'Budget':>10}")
for H in [3, 8, 16, 24, 32, 40, 64]:
    nw = 4*H*(1+H) + 4*H + H + H + H + 1
    r = WeightReader((np.random.randn(nw) * 0.01).tolist())
    m = LSTMModel({'input_size': 1, 'hidden_size': H, 'num_layers': 1}, r)
    m.reset()
    t0 = time.perf_counter()
    m.process(quarter)
    dt = (time.perf_counter() - t0) * 4
    per128 = dt / (SAMPLE_RATE / 128) * 1000
    print(f"{H:>4} {dt*1000:>10.0f} {dt:>8.3f} {per128:>14.2f} {128/SAMPLE_RATE*1000:>10.2f}")

## 10. Summary

In [ ]:
print("=" * 70)
print("NAM Inference — Pure Python+NumPy Feasibility")
print("=" * 70)
print()
for r in results:
    v = 'REAL-TIME OK' if r['rtf'] < 1 else f"{r['rtf']:.0f}x too slow"
    print(f"  {r['name']:<30} {r['avg_ms']:>8.1f} ms  RTF={r['rtf']:.3f}  {v}")
print()
print("""Findings:
  - WaveNet standard (realistic production model) processes 1s in ~38ms
    -> 26x real-time in batch mode, fits in 256-sample buffer budget
    -> BUT: receptive field (4093 samples = 85ms) means you need
       a sliding window of past audio, adding memory/copy overhead
  - LSTM tiny processes 1s in ~670ms (0.67x RTF)
    -> Barely real-time even for the smallest model (H=3)
    -> Real models (H=24+) would be ~700ms+ due to Python loop overhead
    -> The bottleneck is Python's per-sample loop, not matrix math
  - WaveNet is dramatically faster because numpy can batch all samples
    through matrix multiplications, while LSTM must loop sample-by-sample

ConjureDSP Integration:
  - WaveNet models ARE feasible in pure Python+NumPy for buffer sizes >= 256
  - LSTM models are NOT feasible in pure Python — need C/Rust extension
  - Most tone3000 models are WaveNet architecture (good news)
  - Main challenge: managing the receptive field sliding window in
    ConjureDSP's per-buffer process() callback
""")